# Preexisting false merges — case-by-case: does ONE fragment already span 2 GT neurons?

Simple per-case diagnostic for the PREEXISTING false merges the policy made (the err15
category). A merge `merge_labels(A, B)` is flagged PREEXISTING when the two fragments'
dominant GT neurons differ AND at least one fragment ALREADY lands on BOTH neurons (>= `MIN`
nodes each) — i.e. the two-neuron tangle predates the policy, so the gate does not penalize
it (`classify_merge_edits` → `false_preexisting_pairs`).

This notebook does NOT split or re-score anything. For a handful of these errors it just
prints, per fragment, **which GT neurons it already contains and how many nodes on each** —
so you can see the "one fragment is already a multi-neuron tangle" fact directly. Pick which
cases to inspect with `SELECT` below. No image reads, no whole-brain merge scan.

## 0. Setup

In [1]:
import sys, json
from pathlib import Path
import numpy as np, pandas as pd

PROJ = Path("../..").resolve()
sys.path.insert(0, str(PROJ))
sys.path.insert(0, str(PROJ.parent / "segmentation-skeleton-metrics" / "src"))
from proofreader_evolve.harness import incremental_scoring as inc
from proofreader_evolve.harness import dataset as ds
from proofreader_evolve.harness import candidate as cand
from proofreader_evolve.harness.edit_handler import normalize_edits

# --- params -------------------------------------------------------------------
RUN_NAME = "794495_train1_test1_20260715_005246"
GEN      = None      # None -> highest genNN with heuristics.py
BRAIN    = None      # None -> the run's TRAIN brain (has the err15 case); else set e.g. "794491"
SELECT   = 3         # how many preexisting cases to inspect (0 = ALL)
MIN      = inc._MERGE_MIN_NODES     # 50

RUN_DIR = PROJ / "proofreader_evolve" / "runs" / RUN_NAME
_cfg = json.loads((RUN_DIR / "split.json").read_text())
MCL = int(_cfg.get("mcl", 100))
if BRAIN is None:
    BRAIN = str((_cfg.get("train_brains") or _cfg.get("brains") or [""])[0])
if GEN is None:
    gens = sorted(d.name for d in RUN_DIR.glob("gen*") if (d / "heuristics.py").exists())
    GEN = gens[-1] if gens else "gen01"
POLICY = RUN_DIR / GEN / "heuristics.py"
print(f"run={RUN_NAME}  gen={GEN}  brain={BRAIN}  mcl={MCL}  select={SELECT or 'ALL'}")

run=794495_train1_test1_20260715_005246  gen=gen30  brain=794495  mcl=100  select=3


## 1. Load brain + run the policy → its preexisting false merges

In [2]:
prepared = inc.PreparedBrain.load_shared(inc.shared_prepared_cache_path(BRAIN), expect_brain=BRAIN)
frag, _gt, _ = ds.load_cached_graphs(
    ds.default_cache_path(BRAIN, min_cable_length=MCL), expect_brain=BRAIN, expect_mcl=MCL)

# label -> {neuron: node_count} over ALL GT (the fragment→neuron membership map).
full_map = inc.label_gt_counts(prepared, gt_names=None)

# Run the final policy exactly as the run did; keep its merge_labels edits.
pe, mod = cand._load_policy(str(POLICY))
_raw = getattr(mod, "ENUM_PARAMS", None); _raw = {**_raw} if isinstance(_raw, dict) else {}
_raw.setdefault("max_gap_um", 15.0)
enum = ds.resolve_enum_params(_raw)
ssites, msites, _ = cand._enumerate_sites_cached(frag, enum, enumerate_merges=False)
ctx = cand._build_policy_ctx(frag, enum, len(ssites), 0, None)
edits = [e for e in normalize_edits(pe(list(ssites) + list(msites), ctx))
         if e.get("kind") == "merge_labels"]

cls = inc.classify_merge_edits(edits, full_map)
preexisting = cls.get("false_preexisting_pairs", [])   # list of (A, B, da, db)
print(f"{len(edits)} merge edits; false={cls['false']} "
      f"(policy-caused={cls.get('false_policy')}, PREEXISTING={len(preexisting)})")
for k, (A, B, da, db) in enumerate(preexisting):
    print(f"  [{k}] {A} + {B}   (dominant {da} vs {db})")

49837 merge edits; false=59 (policy-caused=6, PREEXISTING=53)
  [0] 21732994834 + 25870679216   (dominant N009-794495-MB vs N007-794495-JG)
  [1] 28022979344 + 23809407743   (dominant N009-794495-MB vs N007-794495-JG)
  [2] 24862427181 + 18587936080   (dominant N023-794495-JG vs N020-794495-JG)
  [3] 22284359662 + 22284359661   (dominant N009-794495-MB vs N019-794495-SP)
  [4] 18587936080 + 22263297706   (dominant N020-794495-JG vs N023-794495-JG)
  [5] 22239795603 + 18587936080   (dominant N023-794495-JG vs N020-794495-JG)
  [6] 21207971077 + 18587936080   (dominant N023-794495-JG vs N020-794495-JG)
  [7] 28738538247 + 25593377852   (dominant N017-794495-IG vs N016-794495-HP)
  [8] 12686451941 + 24744431759   (dominant N006-794495-PP vs N003-794495-SP)
  [9] 10108353125 + 8596021996   (dominant N002-794495-PP vs N001-794495-JT)
  [10] 27680951339 + 27142850899   (dominant N018-794495-YZ vs N008-794495-SP)
  [11] 17527345909 + 18587936080   (dominant N023-794495-JG vs N020-794495-JG)
 

## 2. Case-by-case: per-fragment GT-neuron breakdown

For each selected preexisting error, show BOTH fragments' node counts per GT neuron. A
fragment with >= 2 neurons over the `MIN`-node floor (marked ← TANGLE) is itself a
pre-existing merge — that is the fragment whose entanglement predates the policy's edit.

In [6]:
def show_fragment(label):
    """Print one fragment's GT-neuron node counts; return (n_neurons_over_MIN, dominant)."""
    counts = full_map.get(str(label), {})
    srt = sorted(counts.items(), key=lambda kv: -kv[1])
    over = [n for n, c in srt if c >= MIN]
    tot = sum(counts.values())
    tag = "  <-- TANGLE (already spans >=2 neurons)" if len(over) >= 2 else ""
    print(f"    fragment {label}: {tot} nodes, dominant {srt[0][0] if srt else None}{tag}")
    for n, c in srt:
        flag = " *" if c >= MIN else "  "
        print(f"       {flag} {n:<22} {c:>7}  ({100*c/tot:.1f}%)")
    return len(over), (srt[0][0] if srt else None)

SELECT = 10
cases = preexisting if not SELECT else preexisting[:SELECT]
print(f"Inspecting {len(cases)} of {len(preexisting)} preexisting case(s) on brain {BRAIN}:")
summary = []
for k, (A, B, da, db) in enumerate(cases):
    print(f"\n{'='*72}\nCASE {k}:  merge_labels({A}, {B})   flagged PREEXISTING")
    print(f"  gate saw cross-neuron: dominant({A})={da}  !=  dominant({B})={db}")
    print(f"  {'-'*40}")
    na, _ = show_fragment(A)
    nb, _ = show_fragment(B)
    which = [f for f, n in ((A, na), (B, nb)) if n >= 2]
    verdict = (f"fragment(s) {which} already tangle >=2 neurons -> the merge only EXPOSED a "
               f"pre-existing multi-neuron fragment" if which else
               "neither fragment spans >=2 neurons over MIN (borderline / dominance tie)")
    print(f"  => {verdict}")
    summary.append({"case": k, "label_a": A, "label_b": B, "neuron_a": da, "neuron_b": db,
                    "a_neurons_over_min": na, "b_neurons_over_min": nb,
                    "tangled_fragments": ",".join(which) if which else ""})

summary = pd.DataFrame(summary)
if len(summary):
    print("\n" + "="*72 + "\nSUMMARY"); display(summary)

Inspecting 10 of 53 preexisting case(s) on brain 794495:

CASE 0:  merge_labels(21732994834, 25870679216)   flagged PREEXISTING
  gate saw cross-neuron: dominant(21732994834)=N009-794495-MB  !=  dominant(25870679216)=N007-794495-JG
  ----------------------------------------
    fragment 21732994834: 147428 nodes, dominant N009-794495-MB  <-- TANGLE (already spans >=2 neurons)
        * N009-794495-MB           84356  (57.2%)
        * N007-794495-JG           63072  (42.8%)
    fragment 25870679216: 1883 nodes, dominant N007-794495-JG
        * N007-794495-JG            1883  (100.0%)
  => fragment(s) ['21732994834'] already tangle >=2 neurons -> the merge only EXPOSED a pre-existing multi-neuron fragment

CASE 1:  merge_labels(28022979344, 23809407743)   flagged PREEXISTING
  gate saw cross-neuron: dominant(28022979344)=N009-794495-MB  !=  dominant(23809407743)=N007-794495-JG
  ----------------------------------------
    fragment 28022979344: 673 nodes, dominant N009-794495-MB
      

,case,label_a,label_b,neuron_a,neuron_b,a_neurons_over_min,b_neurons_over_min,tangled_fragments
0,0,21732994834,25870679216,N009-794495-MB,N007-794495-JG,2,1,21732994834
1,1,28022979344,23809407743,N009-794495-MB,N007-794495-JG,1,2,23809407743
2,2,24862427181,18587936080,N023-794495-JG,N020-794495-JG,1,3,18587936080
3,3,22284359662,22284359661,N009-794495-MB,N019-794495-SP,2,2,"22284359662,22284359661"
4,4,18587936080,22263297706,N020-794495-JG,N023-794495-JG,3,1,18587936080
5,5,22239795603,18587936080,N023-794495-JG,N020-794495-JG,1,3,18587936080
6,6,21207971077,18587936080,N023-794495-JG,N020-794495-JG,1,3,18587936080
7,7,28738538247,25593377852,N017-794495-IG,N016-794495-HP,1,2,25593377852
8,8,12686451941,24744431759,N006-794495-PP,N003-794495-SP,2,1,12686451941
9,9,10108353125,8596021996,N002-794495-PP,N001-794495-JT,1,3,8596021996


## 3. (optional) Save the summary table

In [4]:
if len(summary):
    _ana = RUN_DIR / "policy_analysis"; _ana.mkdir(parents=True, exist_ok=True)
    _out = _ana / f"preexisting_case_breakdown_{GEN}_{BRAIN}.csv"
    summary.to_csv(_out, index=False)
    print(f"saved -> {_out}")
else:
    print("nothing to save (no preexisting cases inspected)")

saved -> /allen/programs/mindscope/workgroups/auto-model/zihan.zhang/exaspim-agent/exa-spim-agent/proofreader_evolve/runs/794495_train1_test1_20260715_005246/policy_analysis/preexisting_case_breakdown_gen30_794495.csv
